In [1]:
#install libraries
pip install pandas gensim nltk scikit-learn gtts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.1
    Uninstalling click-8.4.1:
      Successfully uninstalled click-8.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.17.0 requires click>=8.4.0, but you have click 8.1.8 which is incompatible.
typer 0.25.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.27.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [6]:
#import libraries
import pandas as pd
import nltk
import numpy as np
import os

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from gtts import gTTS
from IPython.display import Audio, display

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
#create AWS questions Answer Dataset
data = {
    "question": [
        "What is AWS?",
        "What is EC2 in AWS?",
        "What is S3 in AWS?",
        "What is IAM in AWS?",
        "What is Lambda in AWS?",
        "What is VPC in AWS?",
        "What is RDS in AWS?",
        "What is CloudWatch in AWS?",
        "What is EBS in AWS?",
        "What is Auto Scaling in AWS?"
    ],
    "answer": [
        "AWS stands for Amazon Web Services. It provides cloud computing services like compute, storage, database, networking and security.",
        "EC2 stands for Elastic Compute Cloud. It is used to create and manage virtual servers in AWS.",
        "S3 stands for Simple Storage Service. It is used to store files, images, videos, backups and static website data.",
        "IAM stands for Identity and Access Management. It is used to manage users, groups, roles and permissions securely.",
        "AWS Lambda is a serverless compute service. It runs code without managing servers.",
        "VPC stands for Virtual Private Cloud. It allows us to create a private network inside AWS.",
        "RDS stands for Relational Database Service. It is used to manage databases like MySQL, PostgreSQL, SQL Server and Oracle.",
        "CloudWatch is a monitoring service in AWS. It collects logs, metrics and alarms for AWS resources.",
        "EBS stands for Elastic Block Store. It provides block-level storage for EC2 instances.",
        "Auto Scaling automatically increases or decreases EC2 instances based on application load."
    ]
}
df=pd.DataFrame(data)
df

,question,answer
0,What is AWS?,AWS stands for Amazon Web Services. It provide...
1,What is EC2 in AWS?,EC2 stands for Elastic Compute Cloud. It is us...
2,What is S3 in AWS?,S3 stands for Simple Storage Service. It is us...
3,What is IAM in AWS?,IAM stands for Identity and Access Management....
4,What is Lambda in AWS?,AWS Lambda is a serverless compute service. It...
5,What is VPC in AWS?,VPC stands for Virtual Private Cloud. It allow...
6,What is RDS in AWS?,RDS stands for Relational Database Service. It...
7,What is CloudWatch in AWS?,CloudWatch is a monitoring service in AWS. It ...
8,What is EBS in AWS?,EBS stands for Elastic Block Store. It provide...
9,What is Auto Scaling in AWS?,Auto Scaling automatically increases or decrea...


In [10]:
#tokenize questions
def tokenize(text):
  return nltk.word_tokenize(text.lower())
df["tokens"]=df["question"].apply(tokenize)
df[["question","tokens"]]

,question,tokens
0,What is AWS?,"[what, is, aws, ?]"
1,What is EC2 in AWS?,"[what, is, ec2, in, aws, ?]"
2,What is S3 in AWS?,"[what, is, s3, in, aws, ?]"
3,What is IAM in AWS?,"[what, is, iam, in, aws, ?]"
4,What is Lambda in AWS?,"[what, is, lambda, in, aws, ?]"
5,What is VPC in AWS?,"[what, is, vpc, in, aws, ?]"
6,What is RDS in AWS?,"[what, is, rds, in, aws, ?]"
7,What is CloudWatch in AWS?,"[what, is, cloudwatch, in, aws, ?]"
8,What is EBS in AWS?,"[what, is, ebs, in, aws, ?]"
9,What is Auto Scaling in AWS?,"[what, is, auto, scaling, in, aws, ?]"


In [13]:
#train word2vec model
model=Word2Vec(
    sentences=df["tokens"],
    vector_size=50,
    window=3,
    min_count=1,
    workers=4
)
print("Word2Vec model trained successfully")

Word2Vec model trained successfully


In [17]:
#convert sentence into vector
def sentence_vector(sentence):
  words=tokenize(sentence)
  vectors=[]
  for word in words:
    if word in model.wv:
      vectors.append(model.wv[word])
    if len(vectors)==0:
      return np.zeros(50)
    return np.mean(vectors,axis=0)
question_vectors=np.array([
    sentence_vector(q) for q in df["question"]
])
print(question_vectors.shape)

(10, 50)


In [19]:
#find best answer
def get_answer(user_question):
  user_vector=sentence_vector(user_question).reshape(1,-1)
  similarities=cosine_similarity(user_vector,question_vectors)[0]
  best_index=np.argmax(similarities)
  best_score=similarities[best_index]
  if best_score<0.30:
    return "Sorry, I could not understand your question."
  return df.iloc[best_index]["answer"]

In [26]:
#convert answer text to audio
def speak_colab(text):
    filename = "answer.mp3"
    if os.path.exists(filename):
        os.remove(filename)
    tts = gTTS(text=text, lang="en")
    tts.save(filename)
    display(Audio(filename, autoplay=True))

In [27]:
#test chatbot in colab
user_input=input("Ask AWS question")
answer=get_answer(user_input)
print("Bot:", answer)

speak_colab(answer)

Ask AWS questionwhat is ebs in aws
Bot: EBS stands for Elastic Block Store. It provides block-level storage for EC2 instances.
